## Model Training

In [ ]:
# Set to True to persist data/checkpoints/logs in Google Drive across sessions
# (strongly recommended -- Colab's local disk is wiped whenever the runtime resets).
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/food_classifier"
else:
    BASE_DIR = "./food_classifier"

print(f"Using BASE_DIR = {BASE_DIR}")

## Import Python Modules

In [ ]:
import torch
import time
import copy
import numpy as np
import torch.nn as nn
import torch.optim as optim
from dataclasses import dataclass
from pathlib import Path
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from datetime import datetime
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, Subset
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.datasets import Food101
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Experiment Configuration

In [ ]:
@dataclass
class Config:
    data_dir: str = "./data"
    output_dir: str = f"{BASE_DIR}/checkpoints/"
    logdir: str = f"{BASE_DIR}/runs/"

    batch_size: int = 64
    num_workers: int = 2          # Colab notebooks are often happier with fewer workers
    image_size: int = 224

    epochs_head: int = 3
    epochs_finetune: int = 10
    lr_head: float = 1e-3
    lr_finetune: float = 1e-4
    weight_decay: float = 1e-3
    label_smoothing: float = 0.1
    grad_clip_norm: float = 1.0
    warmup_epochs: int = 2 # linear LR warmup epochs before cosine annealing kicks in, per phase

    # Path to a "latest" checkpoint to resume from, or None to start fresh.
    # e.g. resume = f"{BASE_DIR}/checkpoints/resnet50_food101_latest.pt"
    resume: str = None

    # Override the TensorBoard run folder name. Leave as None to auto-pick:
    # a fresh timestamp for a new run, or the resumed checkpoint's original
    # run name when `resume` is set (so curves continue in the same chart).
    run_name: str = None


CFG = Config()

Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)
Path(CFG.logdir).mkdir(parents=True, exist_ok=True)

checkpoint_path = str(Path(CFG.output_dir) / "resnet50_food101_best.pt")
latest_checkpoint_path = str(Path(CFG.output_dir) / "resnet50_food101_latest.pt")
classes_path = str(Path(CFG.output_dir) / "classes.txt")

NUM_CLASSES = 101

# Imagenet normaliztion stats - required since we are using weights trained on ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(CFG)

## Model Building

### Load Data and Train-Validation split

In this step the training set is spilt into two smaller subsets. The first one is used to train the model and the second is used to evaluate the performance of the model during training and for hyperparameter tuning.

The provided test set remains hidden during the training, evaluation and hyperparameter tuning process and is used as a final unbiased evaluation of the selected model.

In [ ]:
def get_dataloaders(data_dir: str, batch_size: int, num_workers: int, image_size: int = 224):
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.7, 1.0)),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    # No augmentation at eval time -- deterministic resize/crop only.
    eval_transform = transforms.Compose([
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    train_set = Food101(root=data_dir, split="train", transform=train_transform, download=True)
    test_set = Food101(root=data_dir, split="test", transform=eval_transform, download=True)
    train_set_noaug = Food101(root=data_dir, split="train", transform=eval_transform, download=True)

    labels = [train_set._labels[i] for i in range(len(train_set))]
    train_idx, val_idx = train_test_split(
        range(len(train_set)), test_size=0.20, stratify=labels, random_state=42
    )

    train_subset = Subset(train_set, train_idx)
    val_subset = Subset(train_set_noaug, val_idx)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True, drop_last=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)

    return train_loader, test_loader, train_set.classes, val_loader

In [ ]:
train_loader, test_loader, class_names, val_loader = get_dataloaders(
    CFG.data_dir, CFG.batch_size, CFG.num_workers, CFG.image_size
)

Path(classes_path).write_text("\n".join(class_names))
print(
    f"""Train images: {len(train_loader.dataset)} |
    Test images: {len(test_loader.dataset)} |
    Val Images {len(val_loader.dataset)} |
    Classes: {len(class_names)}"""
)


## Evaluation Metrics

Given the fact that our dataset is perfectly balanced some suitable metrics for this case are accuracy and f1 score

- **accuracy**: measures the proportion of correct predictions out of all predictions made
- **f1 score**: the harmonic mean of precision  and  recall. It is suitable for cases where low false positive rate and low false negative rate are of equal importance   

## ResNet50 Classifier (Transfer Learning)

In [ ]:
def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    in_features = model.fc.in_features
    # Dropout before the final Linear helps reduce overfitting, since
    # Food-101 is much smaller/noisier than ImageNet.
    model.fc = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_features, num_classes),
    )
    return model


def set_backbone_trainable(model: nn.Module, trainable: bool):
    """Freezes or unfreezes every parameter except the final `fc` head."""
    for name, param in model.named_parameters():
        if not name.startswith("fc."):
            param.requires_grad = trainable


def freeze_batchnorm_stats(model: nn.Module):
    """
    Puts every BatchNorm layer whose weights are frozen into eval mode.
    `model.train()` puts BatchNorm into training mode regardless of
    requires_grad, so without this, a "frozen" backbone still shifts its
    running statistics batch to batch -- a common cause of noisy loss
    during head-only training.
    """
    for module in model.modules():
        if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            if module.weight is not None and not module.weight.requires_grad:
                module.eval()


def build_scheduler(optimizer, total_epochs: int, warmup_epochs: int = 0):
    """
    Epoch-level LR schedule: `warmup_epochs` of linear warmup (ramping from
    10% of the optimizer's set LR up to 100%) followed by cosine annealing
    for the remaining epochs.

    Warmup mainly helps *training stability* early on -- e.g. right after
    unfreezing the backbone in phase 2, when gradients hitting previously-
    frozen layers can be noisy/large.

    Falls back to plain cosine annealing if warmup_epochs <= 0, and clamps
    warmup_epochs to leave at least 1 epoch for the cosine phase so a short
    phase doesn't get entirely swallowed by warmup.
    """
    warmup_epochs = max(0, min(warmup_epochs, total_epochs - 1))
    if warmup_epochs == 0:
        return CosineAnnealingLR(optimizer, T_max=total_epochs)

    warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine = CosineAnnealingLR(optimizer, T_max=total_epochs - warmup_epochs)
    return SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])


model = build_model(NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)
print(model.fc)

### Checkpointing (save/resume)

Two files get saved to `CFG.output_dir`:
- **`resnet50_food101_best.pt`** — plain model weights, updated whenever validation accuracy improves. Used for inference.
- **`resnet50_food101_latest.pt`** — full resumable state (model + optimizer + scheduler + phase/epoch/TensorBoard counters + run name), overwritten after every epoch. Used for `CFG.resume`.

In [ ]:
def save_checkpoint(path: str, model, optimizer, scheduler, phase: str, epoch: int,
                     global_epoch: int, global_step: int, best_acc: float, run_name: str):
    torch.save({
        "phase": phase,                    # "head" or "finetune"
        "epoch": epoch,                    # last completed epoch *within this phase*
        "global_epoch": global_epoch,
        "global_step": global_step,
        "best_acc": best_acc,
        "run_name": run_name,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
    }, path)


def load_checkpoint(path: str, device: torch.device) -> dict:
    ckpt = torch.load(path, map_location=device)
    required_keys = {"phase", "epoch", "global_epoch", "global_step", "best_acc", "model_state_dict"}
    missing = required_keys - ckpt.keys()
    if missing:
        raise ValueError(f"Checkpoint at {path} is missing expected keys: {missing}")
    return ckpt

### Training Loop Utilties

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, train: bool,
              writer: SummaryWriter = None, global_step: int = 0, grad_clip_norm: float = 1.0):
    """
    One pass over the data. Returns (avg_loss, accuracy, macro_f1, global_step).

    F1 is computed once over the whole epoch's accumulated predictions (not
    averaged per-batch) using macro averaging, so every food class counts
    equally regardless of how visually distinctive or common it is.

    If `writer` is given and train=True, logs every individual batch's loss
    to TensorBoard under "Loss/train_step" - the ground truth for whether
    the model is learning, since the live progress bar and per-epoch average
    can both be too noisy/coarse to judge a trend from by eye.
    """
    model.train() if train else model.eval()
    if train:
        freeze_batchnorm_stats(model)

    total_loss, total_correct, total_samples = 0.0, 0, 0
    all_preds, all_labels = [], []
    context = torch.enable_grad() if train else torch.no_grad()

    with context:
        pbar = tqdm(loader, desc="train" if train else "eval", leave=False)
        for images, labels in pbar:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if train:
                loss.backward()
                # Clips gradient norm so one unlucky/hard batch can't produce
                # an outsized weight update -- a common source of spiky loss.
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
                optimizer.step()
                if writer is not None:
                    writer.add_scalar("Loss/train_step", loss.item(), global_step)
                global_step += 1

            batch_size = images.size(0)
            preds = outputs.argmax(dim=1)
            total_loss += loss.item() * batch_size
            total_correct += (preds == labels).sum().item()
            total_samples += batch_size

            all_preds.append(preds.detach().cpu())
            all_labels.append(labels.detach().cpu())

            pbar.set_postfix(loss=total_loss / total_samples, acc=total_correct / total_samples)

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return total_loss / total_samples, total_correct / total_samples, f1, global_step

In [ ]:
def train_phase(model, train_loader, test_loader, criterion, optimizer, scheduler,
                 device, epochs, phase_name, best_acc, best_state, checkpoint_path,
                 writer: SummaryWriter, global_epoch: int, global_step: int, run_name: str,
                 start_epoch: int = 1, latest_checkpoint_path: str = None, grad_clip_norm: float = 1.0):
    """
    Runs epochs `start_epoch..epochs` of train+eval, tracking and saving the
    best checkpoint. `start_epoch` > 1 when resuming mid-phase.

    `global_epoch`/`global_step` are running counters so phase 1 and phase 2
    (and a resumed continuation of either) land on one continuous x-axis in
    TensorBoard instead of each phase/resume resetting to 0.
    """
    if start_epoch > epochs:
        print(f"[{phase_name}] already completed {epochs} epochs (resumed past this phase) -- skipping")
        return best_acc, best_state, global_epoch, global_step

    for epoch in range(start_epoch, epochs + 1):
        global_epoch += 1
        start = time.time()
        train_loss, train_acc, train_f1, global_step = run_epoch(
            model, train_loader, criterion, optimizer, device, train=True,
            writer=writer, global_step=global_step, grad_clip_norm=grad_clip_norm,
        )
        val_loss, val_acc, val_f1, _ = run_epoch(model, test_loader, criterion, optimizer, device, train=False)
        if scheduler is not None:
            scheduler.step()
        elapsed = time.time() - start

        print(
            f"[{phase_name}] epoch {epoch}/{epochs} "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_f1={train_f1:.4f} "
            f"val_acc={val_acc:.4f} val_f1={val_f1:.4f} "
            f"({elapsed:.1f}s)"
        )

        writer.add_scalar("Loss/train", train_loss, global_epoch)
        writer.add_scalar("Loss/val", val_loss, global_epoch)
        writer.add_scalar("Accuracy/train", train_acc, global_epoch)
        writer.add_scalar("Accuracy/val", val_acc, global_epoch)
        writer.add_scalar("F1/train", train_f1, global_epoch)
        writer.add_scalar("F1/val", val_f1, global_epoch)
        writer.add_scalar("LR", optimizer.param_groups[0]["lr"], global_epoch)

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, checkpoint_path)
            print(f"  -> new best model saved to {checkpoint_path} (val_acc={best_acc:.4f})")

        if latest_checkpoint_path is not None:
            save_checkpoint(
                latest_checkpoint_path, model, optimizer, scheduler,
                phase_name, epoch, global_epoch, global_step, best_acc, run_name,
            )

    return best_acc, best_state, global_epoch, global_step

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {CFG.logdir}

### Model Training

Trains on `train_loader`, evaluates on `val_loader` every epoch to decide which checkpoint is
"best", and only evaluates on `test_loader` once, right at the end, using the best checkpoint. The test set is never involved in any decision made during training.

In [ ]:
best_acc, best_state, global_epoch, global_step = 0.0, None, 0, 0
start_epoch_head, start_epoch_finetune = 1, 1
resume_phase = None
ckpt = None

if CFG.resume:
    print(f"Resuming from checkpoint: {CFG.resume}")
    ckpt = load_checkpoint(CFG.resume, device)
    model.load_state_dict(ckpt["model_state_dict"])
    best_acc = ckpt["best_acc"]
    global_epoch = ckpt["global_epoch"]
    global_step = ckpt["global_step"]
    resume_phase = ckpt["phase"]
    resume_epoch = ckpt["epoch"]

    if resume_phase == "head":
        start_epoch_head = resume_epoch + 1
    elif resume_phase == "finetune":
        start_epoch_head = CFG.epochs_head + 1  # phase 1 already complete -- skip it
        start_epoch_finetune = resume_epoch + 1
    else:
        raise ValueError(f"Unrecognized phase '{resume_phase}' in checkpoint")

    print(f"  Resuming phase='{resume_phase}', last completed epoch={resume_epoch}, "
          f"best_acc so far={best_acc:.4f}")

# Resolve TensorBoard run name: explicit CFG.run_name > resumed checkpoint's
# original run_name (keeps curves continuous) > a fresh timestamp.
if CFG.run_name:
    run_name = CFG.run_name
elif ckpt is not None and ckpt.get("run_name"):
    run_name = ckpt["run_name"]
else:
    run_name = datetime.now().strftime("%Y%m%d_%H%M%S")

log_dir = str(Path(CFG.logdir) / run_name)
writer = SummaryWriter(log_dir=log_dir)
is_resuming_same_run = bool(CFG.resume) and not CFG.run_name and ckpt is not None and ckpt.get("run_name")
print(f"TensorBoard logging to: {log_dir}" + (" (continuing existing run)" if is_resuming_same_run else ""))

if not is_resuming_same_run:
    hparams_text = "\n".join(f"{k}: {v}" for k, v in CFG.__dict__.items())
    writer.add_text("hyperparameters", hparams_text)
    sample_images, _ = next(iter(train_loader))
    writer.add_graph(model, sample_images.to(device))

# ---------- Phase 1: train head only ----------
print("\n=== Phase 1: training classifier head (backbone frozen) ===")
set_backbone_trainable(model, trainable=False)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG.lr_head, weight_decay=CFG.weight_decay,
)
scheduler = build_scheduler(optimizer, CFG.epochs_head, CFG.warmup_epochs)
if CFG.resume and resume_phase == "head":
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if ckpt["scheduler_state_dict"] is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

best_acc, best_state, global_epoch, global_step = train_phase(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, CFG.epochs_head, "head", best_acc, best_state, checkpoint_path,
    writer, global_epoch, global_step, run_name,
    start_epoch=start_epoch_head, latest_checkpoint_path=latest_checkpoint_path,
    grad_clip_norm=CFG.grad_clip_norm,
)

# ---------- Phase 2: fine-tune whole network ----------
print("\n=== Phase 2: fine-tuning full network ===")
set_backbone_trainable(model, trainable=True)
optimizer = optim.AdamW(model.parameters(), lr=CFG.lr_finetune, weight_decay=CFG.weight_decay)
scheduler = build_scheduler(optimizer, CFG.epochs_finetune, CFG.warmup_epochs)

if CFG.resume and resume_phase == "finetune":
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if ckpt["scheduler_state_dict"] is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

best_acc, best_state, global_epoch, global_step = train_phase(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, CFG.epochs_finetune, "finetune", best_acc, best_state, checkpoint_path,
    writer, global_epoch, global_step, run_name,
    start_epoch=start_epoch_finetune, latest_checkpoint_path=latest_checkpoint_path,
    grad_clip_norm=CFG.grad_clip_norm,
)

# ---------- Final, one-time evaluation on the held-out test set ----------
# Loads the BEST checkpoint (highest val accuracy) rather than whatever the
# last epoch happened to produce, and touches the test set exactly once,
# now that every training/model-selection decision is already finalized.
print("\n=== Final evaluation on held-out test set ===")
model.load_state_dict(best_state)
test_loss, test_acc, test_f1, _ = run_epoch(model, test_loader, criterion, None, device, train=False)
print(f"Test accuracy: {test_acc:.4f} | Test F1: {test_f1:.4f} | Test loss: {test_loss:.4f}")
writer.add_scalar("Accuracy/test_final", test_acc, global_epoch)
writer.add_scalar("F1/test_final", test_f1, global_epoch)

writer.add_hparams(
    {k: v for k, v in CFG.__dict__.items() if isinstance(v, (int, float, str))},
    {"best_val_accuracy": best_acc, "final_test_accuracy": test_acc},
)
writer.close()

print(f"\nTraining complete. Best val accuracy: {best_acc:.4f} | Final test accuracy: {test_acc:.4f}")
print(f"Best weights saved to: {checkpoint_path}")